In [ ]:
# ================================================================
# USER CONFIGURATION — set these paths for your environment
# ================================================================
# Root directory for saving anomaly maps — change to your Drive path
# Maps are saved under: {MAPS_SAVE_DIR}/standard/{model_name}/{category}/
MAPS_SAVE_DIR = ''          # e.g. '/content/drive/MyDrive/BachelorsThesis/results/anomaly_maps'
# ================================================================

# Ablation Study

This notebook presents a systematic investigation of the factors driving
performance differences between the three evaluated models.

Investigation 1: AnomalyDINO 2x2 Factorial Ablation
A controlled factorial design isolating the two hypothesised failure
modes of AnomalyDINO on Real-IAD: intra-class viewpoint variation
and cross-category feature contamination. Four conditions are evaluated
on five representative categories.

Investigation 2: Training Compute Equalisation (Dinomaly vs INP-Former)
Dinomaly and INP-Former use fundamentally different training schedules
resulting in a roughly 9x disparity in total image passes. This
investigation equalises compute budgets to assess whether performance
differences reflect architecture or training volume.

Investigation 3: Cross-View Training Data Volume Compensation
The cross-view protocol reduces training data to approximately 2/5
of the standard protocol. This investigation tests whether scaling
the training budget proportionally recovers the performance gap for
Dinomaly and INP-Former.

Five representative categories are used for Investigation 1:
audiojack, pcb, button_battery, usb, toothbrush
Selected to represent diversity in object geometry, defect type,
and detection difficulty across Real-IAD.

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

from google.colab import drive
import sys

drive.mount('/content/drive')

repo_path = '/content/drive/MyDrive/BachelorsThesis'
dataset_root = '/content/drive/MyDrive/datasets/realiad_512'

if not os.path.exists(repo_path):
    !git clone https://github.com/PurpleMono/BachelorsThesis.git {repo_path}
    !git -C {repo_path} submodule update --init
else:
    !git -C {repo_path} pull
    !git -C {repo_path} submodule update --init

!git -C {repo_path}/models/inp_former fetch origin
!git -C {repo_path}/models/inp_former checkout 6041e2b

sys.path.insert(0, repo_path)

!pip install anomalib==2.4.0 ADEval einops colorama timm kornia -q

import torch
import numpy as np
import pandas as pd
import gc
from pathlib import Path

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Results paths — consistent across all notebooks
results_path = f'{repo_path}/results'

# Protocol-specific score paths
std_results = f'{results_path}/standard'
cv_results = f'{results_path}/crossview'
abl_results = f'{results_path}/ablation'

# Anomaly map paths
std_maps = f'{MAPS_SAVE_DIR}/standard'
cv_maps = f'{MAPS_SAVE_DIR}/crossview'
maps_abl1_mm = f'{MAPS_SAVE_DIR}/ablation/investigation1/multiclass_multiview'
maps_abl1_sm = f'{MAPS_SAVE_DIR}/ablation/investigation1/singleclass_multiview'
maps_abl1_ms = f'{MAPS_SAVE_DIR}/ablation/investigation1/multiclass_singleview'
maps_abl1_ss = f'{MAPS_SAVE_DIR}/ablation/investigation1/singleclass_singleview'
maps_abl2 = f'{MAPS_SAVE_DIR}/ablation/investigation2'
maps_abl3 = f'{MAPS_SAVE_DIR}/ablation/investigation3'
abl4_root = f'{abl_results}/investigation4'
abl4_realiad = f'{abl4_root}/real_iad'
abl4_mvtec = f'{abl4_root}/mvtec'
maps_abl4 = f'{MAPS_SAVE_DIR}/ablation/investigation4'

# Create all directories upfront
for path in [
    std_results,
    cv_results,
    f'{abl_results}/investigation1',
    f'{abl_results}/investigation2',
    f'{abl_results}/investigation3',
    f'{results_path}/weights',
    f'{results_path}/figures',
    f'{abl_results}/investigation4',
    std_maps,
    cv_maps,
    maps_abl1_mm,
    maps_abl1_sm,
    maps_abl1_ms,
    maps_abl1_ss,
    maps_abl2,
    maps_abl3,
    abl4_realiad,
    abl4_mvtec,
    maps_abl4,
]:
    os.makedirs(path, exist_ok=True)

print("All results directories ready")

In [ ]:
import zipfile, shutil, os

zip_dir = '/content/drive/MyDrive/datasets/realiad_512/realiad_512'
target_dir = '/content/realiad_512'
os.makedirs(target_dir, exist_ok=True)

# Copy JSON metadata (small — shutil is fine here)
json_src = '/content/drive/MyDrive/datasets/realiad_512/realiad_jsons'
json_dst = '/content/realiad_512/realiad_jsons'
if not os.path.exists(json_dst):
    shutil.copytree(json_src, json_dst)
    print("JSONs copied")
else:
    print("JSONs already present")

# Unzip each category from Drive zips
for f in sorted(os.listdir(zip_dir)):
    if f.endswith('.zip'):
        category = f.replace('.zip', '')
        if not os.path.exists(f'{target_dir}/{category}'):
            print(f"Unzipping {f}...")
            with zipfile.ZipFile(f'{zip_dir}/{f}', 'r') as z:
                z.extractall(target_dir)
        else:
            print(f"Skipping {category} — already present")

dataset_root = target_dir
print(f"\nDataset ready at: {dataset_root}")

In [ ]:
import importlib.util
import pandas as pd
import numpy as np
import gc

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

realiad_utils = load_module("realiad_utils", f"{repo_path}/data/realiad_utils.py")
trainer       = load_module("trainer", f"{repo_path}/models/trainer.py")
metrics       = load_module("metrics", f"{repo_path}/evaluation/metrics.py")

load_realiad_all        = realiad_utils.load_realiad_all
load_realiad_category   = realiad_utils.load_realiad_category
get_crossview_split     = realiad_utils.get_crossview_split  # added

# Dinomaly — now uses official repo via run_inference_dinomaly
train_dinomaly          = trainer.train_dinomaly
run_inference_dinomaly  = trainer.run_inference_dinomaly

# AnomalyDINO — 16-shot multi-class few-shot protocol
train_anomalydino_fewshot = trainer.train_anomalydino_fewshot

# INP-Former
train_inpformer         = trainer.train_inpformer
run_inference_inpformer = trainer.run_inference_inpformer

# Shared inference for AnomalyDINO
run_inference           = trainer.run_inference

# Utility functions
measure_inference_time   = trainer.measure_inference_time
measure_memory_footprint = trainer.measure_memory_footprint
_load_dataset_class      = trainer._load_dataset_class  # added

# Metrics
compute_i_auroc     = metrics.compute_i_auroc
compute_s_auroc     = metrics.compute_s_auroc
compute_all_metrics = metrics.compute_all_metrics
REALIAD_CONFIG      = metrics.REALIAD_CONFIG

# Cross-view utility — defined inline, used in ablation and analysis notebooks
def compute_degradation_ratio(std_auroc, cv_auroc):
    """Percentage degradation from standard to cross-view protocol."""
    return (std_auroc - cv_auroc) / std_auroc * 100

print("All modules loaded")

In [ ]:
results_path = f'{repo_path}/results'

# Load standard protocol baselines
results_din_std = pd.read_csv(f'{results_path}/standard/dinomaly_scores.csv')
results_dino_std = pd.read_csv(f'{results_path}/standard/anomalydino_scores.csv')
results_inp_std = pd.read_csv(f'{results_path}/standard/inpformer_scores.csv')

# Load cross-view protocol baselines
results_din_cv = pd.read_csv(f'{results_path}/crossview/dinomaly_scores.csv')
results_inp_cv = pd.read_csv(f'{results_path}/crossview/inpformer_scores.csv')

i_auroc_din_std = compute_i_auroc(results_din_std)
i_auroc_dino_std = compute_i_auroc(results_dino_std)
i_auroc_inp_std = compute_i_auroc(results_inp_std)
i_auroc_din_cv = compute_i_auroc(results_din_cv)
i_auroc_inp_cv = compute_i_auroc(results_inp_cv)

print("Standard protocol baselines:")
print(f"  Dinomaly:    {i_auroc_din_std:.4f}")
print(f"  AnomalyDINO: {i_auroc_dino_std:.4f}")
print(f"  INP-Former:  {i_auroc_inp_std:.4f}")
print("\nCross-view baselines:")
print(f"  Dinomaly:    {i_auroc_din_cv:.4f}")
print(f"  INP-Former:  {i_auroc_inp_cv:.4f}")

# Investigation 2 INP-Former baseline (22 epochs, 30 categories)
# Used as the 30-category reference for Investigation 4 INP-Former runs
results_inp_inv2 = pd.read_csv(
    f'{abl_results}/investigation2/inpformer_equalised_scores.csv'
)
i_auroc_inp_inv2 = compute_i_auroc(results_inp_inv2)
print(f"\nInvestigation 2 baseline (INP-Former 22 epochs): {i_auroc_inp_inv2:.4f}")

# Load full Real-IAD dataframe for category subsetting
df_all = load_realiad_all(dataset_root)
print(f"Real-IAD dataframe loaded: {len(df_all)} rows")

In [ ]:
# Download MVTec AD if not already present
# Only needed for AnomalyDINO scaling runs in this notebook
if not os.path.exists('/content/datasets/MVTec/bottle'):
    print('Downloading MVTec AD...')
    from anomalib.data import MVTecAD as _dl
    dm = _dl(root='/content/datasets/MVTec', category='bottle')
    dm.prepare_data()
    print('Done')
else:
    print('MVTec AD already present')

## Investigation 1: AnomalyDINO 2x2 Factorial Ablation

AnomalyDINO is evaluated in a 16-shot few-shot setting across all conditions,
using 16 reference images per category per viewpoint consistently.
This matches the standard protocol configuration and keeps the number
of reference images constant so only the factorial factors vary.

Two hypotheses explain the near-random standard protocol I-AUROC:
  H1: Intra-class viewpoint variation — normal patches from different
      angles are dissimilar, producing high distances even for normal images.
  H2: Cross-category contamination — anomalous patches find false nearest
      neighbours in normal features from other categories.

Four conditions on five representative categories isolate each factor:

  Condition A — Multi-Class, Multi-View (16-shot, all 5 viewpoints)
    Copied from standard protocol results. Baseline.

  Condition B — Single-Class, Multi-View (16-shot, all 5 viewpoints)
    One memory bank per category, all viewpoints.
    Removes H2 (contamination), preserves H1 (viewpoint variation).
    Iterated over 5 representative categories independently.

  Condition C — Multi-Class, Single-View (16-shot, C1 only)
    All 30 categories in one memory bank, C1 viewpoint only.
    Removes H1 (viewpoint variation), preserves H2 (contamination).
    16 shots x 30 categories = 480 reference images total.

  Condition D — Single-Class, Single-View (16-shot, C1 only)
    One memory bank per category, C1 viewpoint only.
    Removes both factors — best case for AnomalyDINO.
    Iterated over 5 representative categories independently.

Interpretation:
  B >> A → H2 (contamination) is the dominant failure mode
  C >> A → H1 (viewpoint variation) is the dominant failure mode
  D only strong → both factors required simultaneously

In [ ]:
# Load all 30 categories for multi-class conditions
print("\nLoading all categories for multi-class conditions...")
df_all = load_realiad_all(data_root=dataset_root)
print(f"  Total: {len(df_all)} images across {df_all['category'].nunique()} categories")

In [ ]:
# Free all GPU memory from Dinomaly and INP-Former before AnomalyDINO
# AnomalyDINO requires maximum available system RAM for CPU memory bank consolidation
# Run this immediately before train_anomalydino
torch.cuda.empty_cache()
gc.collect()
print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

In [ ]:
print("="*60)
print("CONDITION B: Single-Class, Multi-View")
print("One category memory bank per category, all 5 viewpoints")
print("="*60)

all_results_cond_b = []
all_categories = df_all['category'].unique()

for cat in all_categories:
    print(f"\nCategory: {cat}")
    cat_df = df_all[df_all['category'] == cat]

    train_df = cat_df[
        (cat_df['split'] == 'train') &
        (cat_df['label'] == 0)
    ].reset_index(drop=True)

    test_df = cat_df[
        cat_df['split'] == 'test'
    ].reset_index(drop=True)

    model = train_anomalydino_fewshot(
        train_df=train_df,
        n_shots=16,
        device='cuda',
        repo_path=repo_path
    )

    results_cat = run_inference(
        model=model,
        test_df=test_df,
        model_name='AnomalyDINO',
        device='cuda',
        batch_size=4,
        repo_path=repo_path,
        save_anomaly_maps=True,
        maps_save_dir=maps_abl1_sm
    )

    i_auroc = compute_i_auroc(results_cat)
    print(f"  {cat} I-AUROC: {i_auroc:.4f}")
    all_results_cond_b.append(results_cat)

    torch.cuda.empty_cache()
    gc.collect()
    del model

results_cond_b = pd.concat(all_results_cond_b, ignore_index=True)
results_cond_b.to_csv(
    f'{abl_results}/investigation1/anomalydino_singleclass_multiview_scores.csv',
    index=False)
print(f"\nCondition B complete")
print(f"Total rows: {len(results_cond_b)}")
print(f"Mean I-AUROC: {compute_i_auroc(results_cond_b):.4f}")

In [ ]:
# Free all GPU memory from Dinomaly and INP-Former before AnomalyDINO
# AnomalyDINO requires maximum available system RAM for CPU memory bank consolidation
# Run this immediately before train_anomalydino
torch.cuda.empty_cache()
gc.collect()
print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

In [ ]:
print("="*60)
print("CONDITION C: Multi-Class, Single-View")
print("All 30 categories, C1 only, 16-shot per category")
print("="*60)

# Build one multi-class memory bank from C1 only
# 16 shots x 30 categories = 480 reference images
train_df_c1_all = df_all[
    (df_all['split'] == 'train') &
    (df_all['label'] == 0) &
    (df_all['viewpoint'] == 'C1')
].reset_index(drop=True)

print("Building multi-class C1-only memory bank...")
model_mc_sv = train_anomalydino_fewshot(
    train_df=train_df_c1_all,
    n_shots=16,
    device='cuda',
    repo_path=repo_path
)

# Inference on all 30 categories, C1 test images only
test_df_c1_all = df_all[
    (df_all['split'] == 'test') &
    (df_all['viewpoint'] == 'C1')
].reset_index(drop=True)

print(f"\nRunning inference on {len(test_df_c1_all)} C1 test images "
      f"across {test_df_c1_all['category'].nunique()} categories")

results_cond_c = run_inference(
    model=model_mc_sv,
    test_df=test_df_c1_all,
    model_name='AnomalyDINO',
    device='cuda',
    batch_size=4,
    repo_path=repo_path,
    save_anomaly_maps=True,
    maps_save_dir=maps_abl1_ms
)

results_cond_c.to_csv(
    f'{abl_results}/investigation1/anomalydino_multiclass_singleview_scores.csv',
    index=False)

torch.cuda.empty_cache()
gc.collect()
del model_mc_sv

print(f"\nCondition C complete")
print(f"Mean I-AUROC: {compute_i_auroc(results_cond_c):.4f}")

In [ ]:
# Free all GPU memory from Dinomaly and INP-Former before AnomalyDINO
# AnomalyDINO requires maximum available system RAM for CPU memory bank consolidation
# Run this immediately before train_anomalydino
torch.cuda.empty_cache()
gc.collect()
print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

In [ ]:
print("="*60)
print("CONDITION D: Single-Class, Single-View")
print("One category memory bank, C1 only, 16-shot, all 30 categories")
print("Best case for AnomalyDINO — both factors removed")
print("="*60)

all_results_cond_d = []

for cat in df_all['category'].unique():
    print(f"\nCategory: {cat}")
    cat_df = df_all[df_all['category'] == cat]

    train_df_c1 = cat_df[
        (cat_df['split'] == 'train') &
        (cat_df['label'] == 0) &
        (cat_df['viewpoint'] == 'C1')
    ].reset_index(drop=True)

    test_df_c1 = cat_df[
        (cat_df['split'] == 'test') &
        (cat_df['viewpoint'] == 'C1')
    ].reset_index(drop=True)

    print(f"  Reference images: {min(16, len(train_df_c1))} | "
          f"Test images: {len(test_df_c1)}")

    model = train_anomalydino_fewshot(
        train_df=train_df_c1,
        n_shots=16,
        device='cuda',
        repo_path=repo_path
    )

    results_cat = run_inference(
        model=model,
        test_df=test_df_c1,
        model_name='AnomalyDINO',
        device='cuda',
        batch_size=4,
        repo_path=repo_path,
        save_anomaly_maps=True,
        maps_save_dir=maps_abl1_ss
    )

    i_auroc = compute_i_auroc(results_cat)
    print(f"  {cat} I-AUROC: {i_auroc:.4f}")
    all_results_cond_d.append(results_cat)

    torch.cuda.empty_cache()
    gc.collect()
    del model

results_cond_d = pd.concat(all_results_cond_d, ignore_index=True)
results_cond_d.to_csv(
    f'{abl_results}/investigation1/anomalydino_singleclass_singleview_scores.csv',
    index=False)
print(f"\nCondition D complete")
print(f"Mean I-AUROC: {compute_i_auroc(results_cond_d):.4f}")

In [ ]:
# Condition A: from standard protocol results
results_dino_std = pd.read_csv(f'{std_results}/anomalydino_scores.csv')

# Copy Condition A to ablation folder for completeness
import shutil
src = f'{std_results}/anomalydino_scores.csv'
dst = f'{abl_results}/investigation1/anomalydino_multiclass_multiview_scores.csv'
if os.path.exists(src):
    shutil.copy(src, dst)
print("Condition A copied from standard protocol results")

# Use all 30 categories
all_categories = sorted(df_all['category'].unique().tolist())

# Build summary table from concatenated result dataframes
abl1_rows = []
for cat in all_categories:
    a = compute_i_auroc(
        results_dino_std[results_dino_std['category'] == cat])
    b = compute_i_auroc(
        results_cond_b[results_cond_b['category'] == cat])
    c = compute_i_auroc(
        results_cond_c[results_cond_c['category'] == cat])
    d = compute_i_auroc(
        results_cond_d[results_cond_d['category'] == cat])
    abl1_rows.append({
        'Category': cat,
        'A: MC+MV (standard)': round(a, 4),
        'B: SC+MV': round(b, 4),
        'C: MC+SV': round(c, 4),
        'D: SC+SV (best case)': round(d, 4),
        'B-A (contamination effect)': round(b - a, 4),
        'C-A (viewpoint effect)': round(c - a, 4),
        'D-A (combined effect)': round(d - a, 4),
    })

abl1_df = pd.DataFrame(abl1_rows)

# Add mean row
means = abl1_df.mean(numeric_only=True)
means['Category'] = 'Mean'
abl1_df = pd.concat(
    [abl1_df, pd.DataFrame([means])], ignore_index=True)

print("="*70)
print("INVESTIGATION 1: AnomalyDINO 2x2 FACTORIAL ABLATION")
print("MC=Multi-Class, SC=Single-Class, MV=Multi-View, SV=Single-View")
print("="*70)
print(abl1_df.round(4).to_string(index=False))

os.makedirs(f'{abl_results}/investigation1', exist_ok=True)
abl1_df.to_csv(
    f'{abl_results}/investigation1/factorial_summary.csv', index=False)
print(f"\nSaved to ablation/investigation1/factorial_summary.csv")

print("\nInterpretation:")
means_only = abl1_df[abl1_df['Category'] != 'Mean']
mean_b_effect = means_only['B-A (contamination effect)'].mean()
mean_c_effect = means_only['C-A (viewpoint effect)'].mean()
print(f"  Mean contamination effect (B-A): {mean_b_effect:+.4f}")
print(f"  Mean viewpoint effect (C-A):     {mean_c_effect:+.4f}")
if abs(mean_c_effect) > abs(mean_b_effect):
    print("  → Viewpoint variation is the dominant failure mode (H1 confirmed)")
else:
    print("  → Cross-category contamination is the dominant failure mode (H2 confirmed)")

## Investigation 2: Training Compute Equalisation

Dinomaly uses 50,000 iterations with batch size 16 resulting in
approximately 800,000 total image passes on Real-IAD.

INP-Former uses 200 epochs resulting in approximately 7,293,000
total image passes — roughly 9x more than Dinomaly.

This investigation trains  INP-Former model at an equalised compute budget
of approximately 800,000 image passes to assess whether observed
performance differences reflect architecture or training volume.

Equalised budgets:
- INP-Former: 22 epochs (~800K image passes)

Both models are evaluated on the standard protocol (all 30 categories,
all 5 viewpoints) to isolate the effect of compute from viewpoint shift.

In [ ]:
print("="*60)
print("INVESTIGATION 2: COMPUTE EQUALISATION (INP-Former only)")
print("="*60)

# Dinomaly: 50,000 iterations x batch 16 = 800,000 image passes
# INP-Former equalised: 800,000 / 36,465 images = ~22 epochs
# Dinomaly excluded from retraining — standard protocol result
# (50k iterations) serves as its baseline directly
EQUALISED_EPOCHS_INPFORMER = 22

# Load full training and test sets
train_df_std = df_all[
    (df_all['split'] == 'train') & (df_all['label'] == 0)]
test_df_std = df_all[df_all['split'] == 'test']

print(f"INP-Former equalised epochs: {EQUALISED_EPOCHS_INPFORMER}")
print(f"(~800,000 image passes — matches Dinomaly standard protocol)")
print(f"Training images: {len(train_df_std)}")

model_inp_eq = train_inpformer(
    train_df=train_df_std,
    dataset_root=dataset_root,
    n_epochs=EQUALISED_EPOCHS_INPFORMER,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{results_path}/weights/inpformer_equalised.pth'
)

results_inp_eq = run_inference_inpformer(
    model=model_inp_eq,
    test_df=test_df_std,
    dataset_root=dataset_root,
    device='cuda',
    batch_size=16,
    repo_path=repo_path,
    save_anomaly_maps=False
)

os.makedirs(results_path, exist_ok=True)
results_inp_eq.to_csv(
    f'{results_path}/inpformer_equalised_scores.csv', index=False)
print(f"Saved: {len(results_inp_eq)} rows")

i_auroc_inp_eq = compute_i_auroc(results_inp_eq)
i_auroc_inp_std = compute_i_auroc(
    pd.read_csv(f'{results_path}/inpformer_standard_scores.csv'))
i_auroc_din_std = compute_i_auroc(
    pd.read_csv(f'{results_path}/dinomaly_standard_scores.csv'))

print(f"\nCOMPUTE EQUALISATION RESULTS")
print(f"Dinomaly standard (50k iter, ~800K passes):    {i_auroc_din_std:.4f}")
print(f"INP-Former standard (200 epochs, ~7.3M passes): {i_auroc_inp_std:.4f}")
print(f"INP-Former equalised (22 epochs, ~800K passes): {i_auroc_inp_eq:.4f}")
print(f"Delta (equalised vs standard): "
      f"{i_auroc_inp_eq - i_auroc_inp_std:+.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_inp_eq

In [ ]:
abl2_summary = pd.DataFrame({
    'Model': ['Dinomaly', 'INP-Former'],
    'Standard I-AUROC': [i_auroc_din_std, i_auroc_inp_std],
    'Standard Image Passes': [800000, 7293000],
    'Equalised I-AUROC': ['N/A (baseline)', i_auroc_inp_eq],
    'Equalised Image Passes': [800000, 1600000],
    'Delta': [
        'N/A',
        round(i_auroc_inp_eq - i_auroc_inp_std, 4),
    ],
})

print("="*70)
print("INVESTIGATION 2: COMPUTE EQUALISATION RESULTS")
print("="*70)
print(abl2_summary.to_string(index=False))

abl2_summary.to_csv(
    f'{abl_results}/investigation2/compute_summary.csv', index=False)
print(f"\nSaved to ablation/investigation2/compute_summary.csv")

## Investigation 3: Cross-View Training Data Volume Compensation (INP-Former)

The cross-viewpoint protocol trains on C1 and C2 only, reducing the
training set to approximately 2/5 of the standard protocol size.

For Dinomaly, this has no effect on total compute — iteration-based
training processes a fixed number of image passes (50,000 iterations
x batch size 16 = 800,000 passes) regardless of dataset size. Any
performance degradation for Dinomaly under the cross-view protocol
is therefore attributable purely to reduced viewpoint diversity, not
reduced training volume.

For INP-Former, epoch-based training means fewer total image passes
when the dataset is smaller:
- Standard protocol: 100 epochs x ~36,465 images = ~3,600,000 passes
- Cross-view protocol: 100 epochs x ~14,586 images = ~1,450,000 passes

This represents a ~2.5x reduction in total image passes for INP-Former
under the cross-view protocol. This investigation compensates by
scaling epochs proportionally:
- Compensated: 250 epochs x ~14,586 images = ~3,600,000 passes

If performance recovers with 500 epochs, the cross-view degradation
for INP-Former is attributable to reduced training volume. If it does
not recover, viewpoint coverage itself is the primary factor.

In [ ]:
# Load cross-view baseline results
results_din_cv = pd.read_csv(f'{cv_results}/dinomaly_scores.csv')
results_inp_cv = pd.read_csv(f'{cv_results}/inpformer_scores.csv')
i_auroc_din_cv = compute_i_auroc(results_din_cv)
i_auroc_inp_cv = compute_i_auroc(results_inp_cv)

# Load standard protocol results
results_din_std = pd.read_csv(f'{std_results}/dinomaly_scores.csv')
results_inp_std = pd.read_csv(f'{std_results}/inpformer_scores.csv')
i_auroc_din_std = compute_i_auroc(results_din_std)
i_auroc_inp_std = compute_i_auroc(results_inp_std)

print("Cross-view baselines:")
print(f"  Dinomaly:    {i_auroc_din_cv:.4f}")
print(f"  INP-Former:  {i_auroc_inp_cv:.4f}")
print("\nStandard protocol baselines:")
print(f"  Dinomaly:    {i_auroc_din_std:.4f}")
print(f"  INP-Former:  {i_auroc_inp_std:.4f}")
print("\nNote: Dinomaly excluded from volume compensation.")
print("Its iteration-based schedule already matches standard")
print("protocol total image passes under cross-view conditions.")

# Load cross-view training data
train_df_cv, test_df_cv = get_crossview_split(
    df_all,
    train_views=['C1', 'C2'],
    test_views=['C1', 'C2', 'C3', 'C4', 'C5']
)
train_df_cv = train_df_cv[
    train_df_cv['label'] == 0].reset_index(drop=True)

n_std = len(df_all[
    (df_all['split'] == 'train') & (df_all['label'] == 0)])
n_cv = len(train_df_cv)

print(f"\nStandard training images: {n_std}")
print(f"Cross-view training images: {n_cv}")
print(f"Ratio: {n_cv/n_std:.2f}")

# Compensated epochs: match standard protocol total image passes
# Standard: 200 epochs x n_std images = total passes
# Compensated: ? epochs x n_cv images = same total passes
std_passes_inp = 100 * n_std
cv_passes_inp = 100 * n_cv
compensated_epochs = round(std_passes_inp / n_cv)

print(f"\nINP-Former image passes:")
print(f"  Standard (100 epochs):   {std_passes_inp:,}")
print(f"  Cross-view (100 epochs): {cv_passes_inp:,}")
print(f"  Compensated epochs needed: {compensated_epochs}")

In [ ]:
# Compute compensated epochs dynamically based on actual dataset sizes
compensated_epochs = round(
    (100 * len(df_all[
        (df_all['split'] == 'train') & (df_all['label'] == 0)
    ])) / len(train_df_cv)
)

print(f"Training INP-Former with {compensated_epochs} epochs")
print(f"(compensates for cross-view dataset reduction)")

model_inp_comp = train_inpformer(
    train_df=train_df_cv,
    dataset_root=dataset_root,
    n_epochs=compensated_epochs,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{results_path}/weights/inpformer_cv_compensated.pth'
)

results_inp_comp = run_inference_inpformer(
    model=model_inp_comp,
    test_df=test_df_cv,
    dataset_root=dataset_root,
    device='cuda',
    batch_size=16,
    repo_path=repo_path,
    save_anomaly_maps=True,
    maps_save_dir=maps_abl3
)

os.makedirs(f'{abl_results}/investigation3', exist_ok=True)
results_inp_comp.to_csv(
    f'{abl_results}/investigation3/inpformer_cv_compensated_scores.csv',
    index=False)
print(f"Saved: {len(results_inp_comp)} rows")

i_auroc_inp_comp = compute_i_auroc(results_inp_comp)
print(f"\nINP-Former standard (100 epochs):             {i_auroc_inp_std:.4f}")
print(f"INP-Former cross-view (100 epochs):           {i_auroc_inp_cv:.4f}")
print(f"INP-Former cross-view ({compensated_epochs} epochs): {i_auroc_inp_comp:.4f}")
print(f"Recovery: {i_auroc_inp_comp - i_auroc_inp_cv:+.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_inp_comp

In [ ]:
# Dinomaly cross-view note — no compensation needed
deg_din = compute_degradation_ratio(i_auroc_din_std, i_auroc_din_cv)
deg_inp = compute_degradation_ratio(i_auroc_inp_std, i_auroc_inp_cv)
deg_inp_comp = compute_degradation_ratio(i_auroc_inp_std, i_auroc_inp_comp)

abl3_summary = pd.DataFrame([
    {
        'Model': 'Dinomaly',
        'Standard': round(i_auroc_din_std, 4),
        'Cross-View (100 epochs equiv)': round(i_auroc_din_cv, 4),
        'Cross-View Compensated': 'N/A (iteration-based)',
        'Degradation (%)': round(deg_din, 2),
        'Recovery after compensation': 'N/A',
        'Note': 'Fixed image passes regardless of dataset size'
    },
    {
        'Model': 'INP-Former',
        'Standard': round(i_auroc_inp_std, 4),
        'Cross-View (100 epochs equiv)': round(i_auroc_inp_cv, 4),
        'Cross-View Compensated': round(i_auroc_inp_comp, 4),
        'Degradation (%)': round(deg_inp, 2),
        'Recovery after compensation': round(
            i_auroc_inp_comp - i_auroc_inp_cv, 4),
        'Note': f'Compensated to {compensated_epochs} epochs'
    },
])

print("=" * 70)
print("INVESTIGATION 3: CROSS-VIEW VOLUME COMPENSATION")
print("=" * 70)
print(abl3_summary.to_string(index=False))

abl3_summary.to_csv(
    f'{abl_results}/investigation3/volume_summary.csv', index=False)
print("\nSaved to ablation/investigation3/volume_summary.csv")

# Investigation 4: Multi-Class Scaling — Effect of Category Count on Detection Performance

This notebook evaluates how image-level detection performance changes as the number of 
categories in the shared model increases. All models use the same backbone (DINOv2-Register 
ViT-Base/14) and identical preprocessing as the standard protocol.

**Real-IAD (Dinomaly + INP-Former):**
- 10 categories: first 10 alphabetically from the 30-category standard protocol set
- 20 categories: first 20 alphabetically
- 30 categories: standard protocol result (already available)

**MVTec AD (AnomalyDINO):**
- 5 categories: first 5 alphabetically from the 15-category set
- 10 categories: first 10 alphabetically
- 15 categories: existing multi-class result from Investigation 1 (notebook 06)

Dinomaly: 50,000 iterations for all configurations (unchanged — iteration-based training 
already processes a fixed compute budget regardless of dataset size).

INP-Former: 22 epochs for all configurations (unchanged — consistent with Investigation 2 
compute-equalised baseline; Investigation 3 established that volume is not the primary driver 
of performance differences).

In [ ]:
# Real-IAD: alphabetical subsets — strictly nested
all_cats_sorted = sorted(df_all['category'].unique().tolist())
cats_10 = all_cats_sorted[:10]
cats_20 = all_cats_sorted[:20]
cats_30 = all_cats_sorted

print(f'10-category subset: {cats_10}')
print(f'20-category subset: {cats_20}')

# MVTec AD: alphabetical subsets — strictly nested
MVTEC_ALL = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill', 'screw',
    'tile', 'toothbrush', 'transistor', 'wood', 'zipper'
]
mvtec_cats_5  = MVTEC_ALL[:5]
mvtec_cats_10 = MVTEC_ALL[:10]

print(f'MVTec 5-category subset:  {mvtec_cats_5}')
print(f'MVTec 10-category subset: {mvtec_cats_10}')

In [ ]:
# Build filtered train/test dataframes for each category subset
# Defined once here and reused across Dinomaly and INP-Former cells

def get_realiad_split(df, categories):
    train_df = df[
        (df['split'] == 'train') &
        (df['label'] == 0) &
        (df['category'].isin(categories))
    ].reset_index(drop=True)
    test_df = df[
        (df['split'] == 'test') &
        (df['category'].isin(categories))
    ].reset_index(drop=True)
    return train_df, test_df

train_df_10, test_df_10 = get_realiad_split(df_all, cats_10)
train_df_20, test_df_20 = get_realiad_split(df_all, cats_20)

base_train_size = len(train_df_30)   # 30-category training set size
iterations_10 = round(50000 * len(train_df_10) / base_train_size)
iterations_20 = round(50000 * len(train_df_20) / base_train_size)


print(f'Dinomaly iterations — 10-cat: {iterations_10}, 20-cat: {iterations_20}, 30-cat: {iterations_30}')

print(f'10-cat — train: {len(train_df_10)}, test: {len(test_df_10)}')
print(f'20-cat — train: {len(train_df_20)}, test: {len(test_df_20)}')

In [ ]:
print('=== Dinomaly — 10 categories ===')

model_dino_10 = train_dinomaly(
    train_df=train_df_10,
    n_iterations=iterations_10,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{results_path}/weights/dinomaly_10cat.pth'
)
results_dino_10 = run_inference_dinomaly(
    model=model_dino_10,
    test_df=test_df_10,
    device='cuda',
    batch_size=16,
    repo_path=repo_path,
)
results_dino_10.to_csv(f'{abl4_realiad}/dinomaly_10cat_scores.csv', index=False)
i_auroc_dino_10 = compute_i_auroc(results_dino_10)
print(f'Dinomaly 10-cat I-AUROC: {i_auroc_dino_10:.4f}')

del model_dino_10
torch.cuda.empty_cache()
gc.collect()

In [ ]:
print('=== Dinomaly — 20 categories ===')

model_dino_20 = train_dinomaly(
    train_df=train_df_20,
    n_iterations=iterations_20,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{results_path}/weights/dinomaly_20cat.pth'
)
results_dino_20 = run_inference_dinomaly(
    model=model_dino_20,
    test_df=test_df_20,
    device='cuda',
    batch_size=16,
    repo_path=repo_path,
)
results_dino_20.to_csv(f'{abl4_realiad}/dinomaly_20cat_scores.csv', index=False)
i_auroc_dino_20 = compute_i_auroc(results_dino_20)
print(f'Dinomaly 20-cat I-AUROC: {i_auroc_dino_20:.4f}')

del model_dino_20
torch.cuda.empty_cache()
gc.collect()

In [ ]:
print('=== INP-Former — 10 categories (22 epochs) ===')

model_inp_10 = train_inpformer(
    train_df=train_df_10,
    dataset_root=dataset_root,
    n_epochs=22,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{results_path}/weights/inpformer_10cat.pth'
)
results_inp_10 = run_inference_inpformer(
    model=model_inp_10,
    test_df=test_df_10,
    dataset_root=dataset_root,
    device='cuda',
    batch_size=16,
    repo_path=repo_path,
    save_anomaly_maps=False,
)
results_inp_10.to_csv(f'{abl4_realiad}/inpformer_10cat_scores.csv', index=False)
i_auroc_inp_10 = compute_i_auroc(results_inp_10)
print(f'INP-Former 10-cat I-AUROC: {i_auroc_inp_10:.4f}')

del model_inp_10
torch.cuda.empty_cache()
gc.collect()

In [ ]:
print('=== INP-Former — 20 categories (22 epochs) ===')

model_inp_20 = train_inpformer(
    train_df=train_df_20,
    dataset_root=dataset_root,
    n_epochs=22,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{results_path}/weights/inpformer_20cat.pth'
)
results_inp_20 = run_inference_inpformer(
    model=model_inp_20,
    test_df=test_df_20,
    dataset_root=dataset_root,
    device='cuda',
    batch_size=16,
    repo_path=repo_path,
    save_anomaly_maps=False,
)
results_inp_20.to_csv(f'{abl4_realiad}/inpformer_20cat_scores.csv', index=False)
i_auroc_inp_20 = compute_i_auroc(results_inp_20)
print(f'INP-Former 20-cat I-AUROC: {i_auroc_inp_20:.4f}')

del model_inp_20
torch.cuda.empty_cache()
gc.collect()

In [ ]:
from anomalib.models import AnomalyDINO
from anomalib.data import MVTecAD
from torchvision.transforms import v2 as T
from sklearn.metrics import roc_auc_score

transform_mvtec = T.Compose([
    T.Resize((448, 448)),
    T.CenterCrop(392),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

N_SHOTS = 16

def build_anomalydino_bank(categories):
    model = AnomalyDINO(
        encoder_name='dinov2reg_vit_base_14',
        coreset_subsampling=False,
        masking=False,
    )
    tm = model.model.to('cuda')
    tm.train()
    for cat in categories:
        dm = MVTecAD(
            root='/content/datasets/MVTec',
            category=cat,
            train_batch_size=64,
            eval_batch_size=32,
            num_workers=2,
            augmentations=transform_mvtec,
        )
        dm.setup()
        all_train = torch.cat([b['image'] for b in dm.train_dataloader()], dim=0)
        ref = all_train[:N_SHOTS].to('cuda')
        with torch.no_grad():
            tm(ref)
        print(f'  {cat}: {N_SHOTS} shots added')
    tm.embedding_store = [e.cpu() for e in tm.embedding_store]
    torch.cuda.empty_cache()
    tm.to('cpu')
    tm.fit()
    tm.to('cuda')
    tm.memory_bank = tm.memory_bank.to('cuda')
    model.model = tm
    return model

def run_anomalydino_mvtec(model, categories):
    rows = []
    model.model.eval()
    for cat in categories:
        dm = MVTecAD(
            root='/content/datasets/MVTec',
            category=cat,
            eval_batch_size=32,
            num_workers=2,
            augmentations=transform_mvtec,
        )
        dm.setup()
        scores, labels = [], []
        with torch.no_grad():
            for batch in dm.test_dataloader():
                out = model(batch['image'].to('cuda'))
                scores.extend(out.pred_score.cpu().numpy().flatten())
                labels.extend(batch.gt_label.cpu().numpy().flatten())
        auroc = roc_auc_score(labels, scores) * 100
        print(f'  {cat}: {auroc:.2f}%')
        rows.append({'category': cat, 'i_auroc': round(auroc, 2)})
    return pd.DataFrame(rows)

print('AnomalyDINO helper functions ready')

In [ ]:
from anomalib.models import AnomalyDINO
from anomalib.data import MVTecAD
from torchvision.transforms import v2 as T
from sklearn.metrics import roc_auc_score

transform_mvtec = T.Compose([
    T.Resize((448, 448)),
    T.CenterCrop(392),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

N_SHOTS = 16

def build_anomalydino_bank(categories):
    model = AnomalyDINO(
        encoder_name='dinov2reg_vit_base_14',
        coreset_subsampling=False,
        masking=False,
    )
    tm = model.model.to('cuda')
    tm.train()
    for cat in categories:
        dm = MVTecAD(
            root='/content/datasets/MVTec',
            category=cat,
            train_batch_size=64,
            eval_batch_size=32,
            num_workers=2,
            augmentations=transform_mvtec,
        )
        dm.setup()
        all_train = torch.cat([b['image'] for b in dm.train_dataloader()], dim=0)
        ref = all_train[:N_SHOTS].to('cuda')
        with torch.no_grad():
            tm(ref)
        print(f'  {cat}: {N_SHOTS} shots added')
    tm.embedding_store = [e.cpu() for e in tm.embedding_store]
    torch.cuda.empty_cache()
    tm.to('cpu')
    tm.fit()
    tm.to('cuda')
    tm.memory_bank = tm.memory_bank.to('cuda')
    model.model = tm
    return model

def run_anomalydino_mvtec(model, categories):
    rows = []
    model.model.eval()
    for cat in categories:
        dm = MVTecAD(
            root='/content/datasets/MVTec',
            category=cat,
            eval_batch_size=32,
            num_workers=2,
            augmentations=transform_mvtec,
        )
        dm.setup()
        scores, labels = [], []
        with torch.no_grad():
            for batch in dm.test_dataloader():
                out = model(batch['image'].to('cuda'))
                scores.extend(out.pred_score.cpu().numpy().flatten())
                labels.extend(batch.gt_label.cpu().numpy().flatten())
        auroc = roc_auc_score(labels, scores) * 100
        print(f'  {cat}: {auroc:.2f}%')
        rows.append({'category': cat, 'i_auroc': round(auroc, 2)})
    return pd.DataFrame(rows)

print('AnomalyDINO helper functions ready')

In [ ]:
print('=== AnomalyDINO — MVTec 5 categories ===')

model_ad_5 = build_anomalydino_bank(mvtec_cats_5)
results_ad_5 = run_anomalydino_mvtec(model_ad_5, mvtec_cats_5)
results_ad_5.to_csv(f'{abl4_mvtec}/anomalydino_5cat_scores.csv', index=False)
print(f'Mean I-AUROC 5-cat: {results_ad_5["i_auroc"].mean():.2f}%')

del model_ad_5
torch.cuda.empty_cache()
gc.collect()

In [ ]:
print('=== AnomalyDINO — MVTec 10 categories ===')

model_ad_10 = build_anomalydino_bank(mvtec_cats_10)
results_ad_10 = run_anomalydino_mvtec(model_ad_10, mvtec_cats_10)
results_ad_10.to_csv(f'{abl4_mvtec}/anomalydino_10cat_scores.csv', index=False)
print(f'Mean I-AUROC 10-cat: {results_ad_10["i_auroc"].mean():.2f}%')

del model_ad_10
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Load existing 15-cat AnomalyDINO result from notebook 06
results_ad_15 = pd.read_csv(
    f'{results_path}/mvtec_validation_anomalydino_multiclass.csv'
)
i_auroc_ad_15 = results_ad_15['Multi-class I-AUROC'].mean()

print('=' * 60)
print('Investigation 4: Multi-Class Category Scaling')
print('=' * 60)

print(f'\nDinomaly (Real-IAD, 50K iterations):')
print(f'  10 categories: {i_auroc_dino_10:.4f}')
print(f'  20 categories: {i_auroc_dino_20:.4f}')
print(f'  30 categories: {i_auroc_din_std:.4f}  (standard protocol)')

print(f'\nINP-Former (Real-IAD, 22 epochs):')
print(f'  10 categories: {i_auroc_inp_10:.4f}')
print(f'  20 categories: {i_auroc_inp_20:.4f}')
print(f'  30 categories: {i_auroc_inp_inv2:.4f}  (Investigation 2 baseline)')

print(f'\nAnomalyDINO (MVTec AD, 16-shot multi-class):')
print(f'   5 categories: {results_ad_5["i_auroc"].mean():.2f}%')
print(f'  10 categories: {results_ad_10["i_auroc"].mean():.2f}%')
print(f'  15 categories: {i_auroc_ad_15:.2f}%  (Investigation 1 result)')

summary = pd.DataFrame([
    {'model': 'Dinomaly',    'dataset': 'Real-IAD',  'n_categories': 10, 'i_auroc': round(i_auroc_dino_10, 4)},
    {'model': 'Dinomaly',    'dataset': 'Real-IAD',  'n_categories': 20, 'i_auroc': round(i_auroc_dino_20, 4)},
    {'model': 'Dinomaly',    'dataset': 'Real-IAD',  'n_categories': 30, 'i_auroc': round(i_auroc_din_std, 4)},
    {'model': 'INP-Former',  'dataset': 'Real-IAD',  'n_categories': 10, 'i_auroc': round(i_auroc_inp_10, 4)},
    {'model': 'INP-Former',  'dataset': 'Real-IAD',  'n_categories': 20, 'i_auroc': round(i_auroc_inp_20, 4)},
    {'model': 'INP-Former',  'dataset': 'Real-IAD',  'n_categories': 30, 'i_auroc': round(i_auroc_inp_inv2, 4)},
    {'model': 'AnomalyDINO', 'dataset': 'MVTec AD',  'n_categories': 5,  'i_auroc': round(results_ad_5['i_auroc'].mean(), 4)},
    {'model': 'AnomalyDINO', 'dataset': 'MVTec AD',  'n_categories': 10, 'i_auroc': round(results_ad_10['i_auroc'].mean(), 4)},
    {'model': 'AnomalyDINO', 'dataset': 'MVTec AD',  'n_categories': 15, 'i_auroc': round(i_auroc_ad_15, 4)},
])
summary.to_csv(f'{abl4_root}/investigation4_summary.csv', index=False)
print('\nSaved: investigation4_summary.csv')

## Combined Ablation Summary

Synthesises findings across all three investigations.

In [ ]:
print("=" * 70)
print("ABLATION STUDY COMBINED SUMMARY")
print("=" * 70)

print("\nInvestigation 1: AnomalyDINO 2x2 Factorial")
print(abl1_df.round(4).to_string(index=False))

print("\nInvestigation 2: Compute Equalisation (INP-Former only)")
print(abl2_summary.to_string(index=False))

print("\nInvestigation 3: Cross-View Volume Compensation (INP-Former only)")
print(abl3_summary.to_string(index=False))

print("\nInvestigation 4: Multi-Class Scalability-Test")
print(abl4_summary.to_string(index=False))
